# Compare two simulation runs

Point this at two sets of results and it reports, measure by measure, whether they agree —
and when they do not, whether the difference is bigger than the run's own seed-to-seed
noise.

**Intended use:** a previous iteration vs. the current one. The helpers are defined once,
near the top; every cell after them is a thin call. Add, remove, or reorder those freely.

Sections:

1. **Inventory** — what is in each run, and do the measure lists match
2. **CRN diagnostics** — *are these runs supposed to reproduce bit-for-bit?* Run this first
   when `seeds_exact` is 0 but you expected exact agreement
3. **Sweep** — every sim x vehicle x location at once
4. **Single-pair detail** — headline table, shift-vs-noise plot, per-seed plot
5. **Stratified drill-down**

## Parameters

Tagged `parameters` for papermill, matching the other notebooks here.

`SIMS` maps a sim name to its results root on each side, so **both maternal and child are
compared** — the old and new runs nest their outputs differently, which is why each side
gets its own base path rather than one shared template.

Each path may point at a directory of flat `<measure>.parquet` files (older psimulate), a
directory of `<measure>/<hash>.parquet` shards (newer psimulate), or a psimulate `-o`
directory, in which case the newest `<timestamp>/results/` below it is used.

In [ ]:
OLD_ROOT = ("/mnt/team/simulation_science/pub/models/vivarium_gates_lsff_by_wealth_quintile"
            "/2026_08_13_13_55_48")
NEW_ROOT = "/mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model1.0"

LABEL_A = "old (2026_08_13)"
LABEL_B = "new (model1.0)"

# sim -> results base on each side. Extend or trim to taste.
SIMS = {
    "maternal": {"a": f"{OLD_ROOT}/0200_pregnancy_sim/sim_results", "b": f"{NEW_ROOT}/maternal"},
    "child":    {"a": f"{OLD_ROOT}/0300_child_sim/sim_results",     "b": f"{NEW_ROOT}/child"},
}
# (vehicle, location) pairs that exist for these sims.
COMBOS = [("rice", "nigeria"), ("rice", "india"), ("bouillon", "nigeria")]

# Default single pair for the detail sections.
DEFAULT_SIM, DEFAULT_COMBO = "maternal", ("rice", "nigeria")

# Measures renamed between the two runs: {name_in_A: name_in_B}.
MEASURE_ALIASES = {"person_time": "person_time_population"}

# Column holding the intervention scenario. Maternal uses "scenario"; child uses
# "maternal_scenario". None -> auto-detect per frame.
SCENARIO_COL = None

# Restrict comparisons to one scenario so a scenario mix cannot mask a shift.
# None -> total across all scenarios.
SCENARIO = "baseline"

# |Welch t| on per-seed totals above which a measure is flagged.
T_REVIEW = 2.0
T_DIFFERS = 3.0

## Setup

In [ ]:
import difflib
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

try:
    from scipy import stats as _scipy_stats
except ImportError:
    _scipy_stats = None

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", message=".*observed=False.*")

# Categorical slots 1 and 2 of the validated palette. Two series, fixed order: A is always
# blue and B always orange, whatever is being plotted.
COLOR_A, COLOR_B = "#2a78d6", "#eb6834"
# Status palette -- never reused as a series colour, always paired with a text label.
STATUS_COLORS = {"IDENTICAL": "#0ca30c", "OK": "#0ca30c", "REVIEW": "#fab219",
                 "DIFFERS": "#d03b3b", "N/A": "#8a8880"}
INK, INK_MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": "#d8d6d0", "axes.labelcolor": INK_MUTED,
    "axes.titlecolor": INK, "axes.titlesize": 11,
    "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": "#e8e6e0", "grid.linewidth": 0.8,
    "text.color": INK, "xtick.color": INK_MUTED, "ytick.color": INK_MUTED,
    "legend.frameon": False, "font.size": 10, "figure.dpi": 110,
})


def run_paths(sim: str, combo: tuple[str, str]) -> tuple[str, str]:
    """(run_A, run_B) for one sim and (vehicle, location)."""
    vehicle, location = combo
    return (f"{SIMS[sim]['a']}/{vehicle}/{location}",
            f"{SIMS[sim]['b']}/{vehicle}/{location}")


RUN_A, RUN_B = run_paths(DEFAULT_SIM, DEFAULT_COMBO)

### Loading

`resolve_results_dir` absorbs the layout differences so nothing downstream cares which
psimulate version produced a run — `pd.read_parquet` concatenates a directory of shards.

In [ ]:
def _measure_names(d: Path) -> set[str]:
    """Measure names directly under ``d``, from either layout."""
    flat = {f.stem for f in d.glob("*.parquet")}
    sharded = {s.name for s in d.iterdir() if s.is_dir() and any(s.glob("*.parquet"))}
    return flat | sharded


def resolve_results_dir(path: str | Path) -> Path:
    """The directory that directly contains the measures.

    Accepts a flat results dir, a sharded results dir, or a psimulate ``-o`` directory (in
    which case the newest ``<timestamp>/results`` beneath it wins).
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(p)
    if _measure_names(p):
        return p
    for pattern in ("*/results", "*/*/results"):
        nested = sorted(p.glob(pattern))
        if nested:
            return nested[-1]
    raise FileNotFoundError(f"no measures and no */results under {p}")


def list_measures(run: str | Path) -> set[str]:
    return _measure_names(resolve_results_dir(run))


def load_measure(run: str | Path, measure: str) -> pd.DataFrame:
    d = resolve_results_dir(run)
    flat, sharded = d / f"{measure}.parquet", d / measure
    if flat.exists():
        return pd.read_parquet(flat)
    if sharded.is_dir():
        return pd.read_parquet(sharded)
    raise FileNotFoundError(f"{measure} not found in {d}")


def describe_run(run: str | Path, label: str) -> pd.Series:
    d = resolve_results_dir(run)
    measures = sorted(_measure_names(d))
    layout = "flat" if (d / f"{measures[0]}.parquet").exists() else "sharded"
    n_shards = len(list((d / measures[0]).glob("*.parquet"))) if layout == "sharded" else 1
    return pd.Series({"label": label, "path": str(d), "layout": layout,
                      "n_measures": len(measures), "shards_per_measure": n_shards})

## 1. Inventory

In [ ]:
info = pd.DataFrame([describe_run(RUN_A, LABEL_A), describe_run(RUN_B, LABEL_B)])
display(info.set_index("label").T)

meas_a, meas_b = list_measures(RUN_A), list_measures(RUN_B)
aliased = {MEASURE_ALIASES.get(m, m) for m in meas_a}
print(f"measures in both: {len(aliased & meas_b)}")
only_a = sorted(m for m in meas_a if MEASURE_ALIASES.get(m, m) not in meas_b)
only_b = sorted(meas_b - aliased)
if only_a:
    print(f"only in {LABEL_A}: {only_a}")
if only_b:
    print(f"only in {LABEL_B}: {only_b}")
if not (only_a or only_b):
    print("measure inventories match exactly")
if MEASURE_ALIASES:
    print(f"applied renames: {MEASURE_ALIASES}")

## 2. CRN diagnostics — should these runs reproduce exactly?

Vivarium's common-random-numbers machinery is fully deterministic. Reproducibility rests on
a chain of **string** inputs:

* `IndexMap._hash(keys, salt=clock_time)` maps the `randomness.key_columns` (here
  `entrance_time`, `age`) to a position in the random stream.
* `RandomnessStream._key()` builds `"_".join([stream_key, str(clock()), str(additional_key),
  str(seed)])`, SHA1s it, and seeds a `numpy.random.RandomState`.

So two runs agree **bit-for-bit** only if all of these match: the key-column values, the
`str()` of the clock, every stream's key (its component's name), the seeds, and the hash
implementations themselves. Change any one — including a purely cosmetic change to a
component name or to how the clock stringifies — and every draw moves, with no change in
the science.

If `seeds_exact` is 0 across the board and you expected exact agreement, the cause is
almost certainly global (a re-seed), not per-measure. The cells below compare the
environment and configuration psimulate recorded alongside each run.

**Reading the component diff.** The monorepo migration renamed the *import namespace*
(`vivarium_public_health:` -> `vivarium.public_health:`), so that line will differ. On its
own it is **not** a CRN explanation: a stream's key is the component's `name`, which vivarium
derives from the class and its arguments, not from the module it was imported from. What
would matter is a change to the component *entries* — one renamed, added, removed, or moved
to a different position in the list.

**If all three diagnostics come back clean** — same randomness config, same component
entries, and only expected package bumps — then the re-seed happened *inside* vivarium. The
seed for every draw is `sha1("_".join([stream_key, str(clock()), str(additional_key),
str(seed)]))`, and the CRN alignment additionally depends on `IndexMap._hash` over the
`key_columns`. A change to any of those — how the clock stringifies, the hash internals, the
key format — re-randomizes every run while leaving the science untouched.

The decisive experiment at that point is cheap and does not need the old stack: **run the
same new-stack simulation twice with the same seed.** If those two agree bit-for-bit, CRN is
working within the version and the disagreement is purely cross-version, which makes the
per-measure shifts re-randomization artifacts to be judged distributionally (more seeds),
not bugs to be hunted in the model code.

In [ ]:
def _find_run_file(run: str | Path, name: str, verbose: bool = True) -> Path | None:
    """Locate a file psimulate recorded alongside a run.

    Where it lands depends on how psimulate was invoked, so three places are searched:

    * the results directory itself;
    * its parent -- the ``<timestamp>/`` run directory, when results live in
      ``<timestamp>/results/`` (the current layout);
    * one level *down*, in timestamped subdirectories. The 0200/0300 Snakefiles run
      psimulate with ``-o ..`` from inside ``sim_results/{vehicle}/{location}`` and then
      ``mv ./*/results/*.parquet .``, which hoists the parquets up but leaves
      ``model_specification.yaml``, ``requirements.txt`` and the logs behind in
      ``sim_results/{vehicle}/{location}/<timestamp>/``.

    The newest match wins, so a restart's directory beats the original run's.
    """
    d = resolve_results_dir(run)
    searched = []
    for cand in (d, d.parent):
        searched.append(cand)
        f = cand / name
        if f.exists():
            return f
    # One level down: <timestamp>/ subdirectories left behind by the Snakefile's mv.
    below = sorted((sub / name for sub in d.iterdir() if sub.is_dir()), reverse=True)
    searched.append(d / "*/")
    for f in below:
        if f.exists():
            return f
    if verbose:
        print(f"  (no {name} in: " + ", ".join(str(x) for x in searched) + ")")
    return None


def compare_requirements(run_a=None, run_b=None, only_changed=True) -> pd.DataFrame:
    """Package-version delta between two runs, from psimulate's requirements.txt."""
    run_a, run_b = run_a or RUN_A, run_b or RUN_B

    def parse(run):
        f = _find_run_file(run, "requirements.txt")
        if f is None:
            return None, None
        pkgs = {}
        for line in f.read_text().splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            m = re.split(r"==|@|\s+", line, maxsplit=1)
            if len(m) == 2:
                pkgs[m[0].strip().lower().replace("_", "-")] = m[1].strip()
        return pkgs, f

    a, fa = parse(run_a)
    b, fb = parse(run_b)
    if a is None or b is None:
        print(f"requirements.txt not found for {'A' if a is None else 'B'} "
              "-- older runs may not have recorded one")
        return pd.DataFrame()
    print(f"A: {fa}\nB: {fb}")
    rows = []
    for pkg in sorted(set(a) | set(b)):
        va, vb = a.get(pkg, "(absent)"), b.get(pkg, "(absent)")
        if only_changed and va == vb:
            continue
        rows.append({"package": pkg, LABEL_A: va, LABEL_B: vb})
    out = pd.DataFrame(rows)
    # Surface the packages that actually own the randomness implementation.
    if not out.empty:
        key = out.package.str.contains("vivarium|numpy|pandas", regex=True)
        out = pd.concat([out[key], out[~key]])
    return out.reset_index(drop=True)


compare_requirements()

In [ ]:
def _extract_block(text: str, key: str) -> str:
    """Pull the ``key:`` block out of a YAML document textually.

    The model specification psimulate records is not safely parseable -- it serialises
    live component objects as ``!!python/object:...``, including recursive anchors, which
    safe_load rejects and unsafe_load would execute. A diff only needs the text, so the
    block is sliced by indentation instead of parsed.
    """
    lines = text.splitlines()
    out, base = [], None
    for line in lines:
        stripped = line.strip()
        if base is None:
            if stripped.startswith(f"{key}:"):
                base = len(line) - len(line.lstrip())
                out.append(line)
            continue
        if not stripped:
            out.append(line)
            continue
        if (len(line) - len(line.lstrip())) <= base:
            break
        out.append(line)
    return chr(10).join(out).rstrip() if out else ""


def _diff(a_txt: str, b_txt: str, what: str, normalize: bool = False) -> None:
    """Print a unified diff.

    ``normalize`` compares only the sequence of non-blank, stripped lines. The two runs
    come from different psimulate versions, which may dump YAML with different indentation
    or key order -- a formatting-only diff is a false positive when what you care about is
    whether the *content* changed.
    """
    if normalize:
        a_txt = chr(10).join(l.strip() for l in a_txt.splitlines() if l.strip())
        b_txt = chr(10).join(l.strip() for l in b_txt.splitlines() if l.strip())
    if a_txt == b_txt:
        print(f"{what} is IDENTICAL between the two runs:" + chr(10))
        print(a_txt or "(empty)")
        return
    print(f"{what} DIFFERS:" + chr(10))
    for line in difflib.unified_diff(a_txt.splitlines(), b_txt.splitlines(),
                                     fromfile=LABEL_A, tofile=LABEL_B, lineterm=""):
        print(line)


def compare_configuration(run_a=None, run_b=None, section: str | None = "randomness"):
    """Diff a block of the model_specification.yaml psimulate recorded for each run.

    ``randomness`` is the block that governs CRN alignment (``key_columns``, ``map_size``,
    ``random_seed``). Pass ``section=None`` to diff the whole file.
    """
    run_a, run_b = run_a or RUN_A, run_b or RUN_B
    texts = {}
    for label, run in ((LABEL_A, run_a), (LABEL_B, run_b)):
        f = _find_run_file(run, "model_specification.yaml")
        if f is None:
            print(f"model_specification.yaml not found for {label} "
                  "-- older runs may not have recorded one beside their results")
            return
        raw = f.read_text()
        texts[label] = _extract_block(raw, section) if section else raw
        print(f"{label}: {f}")
    print()
    _diff(texts[LABEL_A], texts[LABEL_B],
          f"configuration block '{section or 'ALL'}'", normalize=True)


compare_configuration(section="randomness")

In [ ]:
# The component list is the other place a global re-seed hides: a RandomnessStream's key is
# its component's name, so a renamed or reordered component changes every draw it makes.
def compare_components(run_a=None, run_b=None):
    """Diff the component list.

    A RandomnessStream's key is its component's name, so a renamed, added, removed or
    reordered component changes every draw that component makes -- with no change to the
    science. This is the most common cause of a whole run failing to reproduce.
    """
    run_a, run_b = run_a or RUN_A, run_b or RUN_B
    texts = {}
    for label, run in ((LABEL_A, run_a), (LABEL_B, run_b)):
        f = _find_run_file(run, "model_specification.yaml")
        if f is None:
            print(f"model_specification.yaml not found for {label}")
            return
        texts[label] = _extract_block(f.read_text(), "components")
    _diff(texts[LABEL_A], texts[LABEL_B], "component list", normalize=True)


compare_components()

## 3. Comparison helpers

`compare_measure` reduces each run to **one total per random seed**, then applies Welch's
t-test to those two samples. Per-seed totals are the right unit: seeds are the replication
mechanism, so their spread *is* the run's noise floor.

| verdict | meaning |
|---|---|
| `IDENTICAL` | every per-seed total matches exactly |
| `OK` | shift small relative to seed noise (`|t| < T_REVIEW`) |
| `REVIEW` | `T_REVIEW <= |t| < T_DIFFERS` |
| `DIFFERS` | `|t| >= T_DIFFERS` |

Two cautions when reading this. `t` measures whether a shift is distinguishable from noise
*at the number of seeds you ran* — more seeds will promote real-but-small shifts. And with
CRN intact you should expect `IDENTICAL`, not `OK`: an `OK` verdict everywhere means the
runs are statistically consistent but **not** reproducing, which is a separate finding.
Check `seeds_exact`, then section 2.

In [ ]:
def _scenario_col(df: pd.DataFrame) -> str | None:
    if SCENARIO_COL:
        return SCENARIO_COL if SCENARIO_COL in df.columns else None
    for c in ("scenario", "maternal_scenario"):
        if c in df.columns:
            return c
    return None


def _per_seed(df: pd.DataFrame, scenario: str | None) -> tuple[pd.Series, str]:
    """Total per random seed, optionally within one scenario.

    Most measures carry a ``value`` column and are summed. Line-list style outputs (the
    maternal ``births`` table) have none; those are compared on row count. The metric is
    returned so a count is never silently compared against a value sum.
    """
    col = _scenario_col(df)
    if scenario is not None and col is not None:
        df = df[df[col] == scenario]
    metric = "sum(value)" if "value" in df.columns else "row count"
    if "random_seed" not in df.columns:
        total = df["value"].sum() if "value" in df.columns else float(len(df))
        return pd.Series({-1: total}), metric
    g = df.groupby("random_seed")
    return (g["value"].sum() if "value" in df.columns else g.size().astype(float)), metric


def compare_measure(measure_a, measure_b=None, scenario="__default__",
                    run_a=None, run_b=None) -> dict:
    run_a, run_b = run_a or RUN_A, run_b or RUN_B
    measure_b = measure_b or MEASURE_ALIASES.get(measure_a, measure_a)
    scenario = SCENARIO if scenario == "__default__" else scenario
    row = {"measure": measure_a, "scenario": scenario or "(all)"}
    try:
        a, metric_a = _per_seed(load_measure(run_a, measure_a), scenario)
        b, metric_b = _per_seed(load_measure(run_b, measure_b), scenario)
    except Exception as exc:  # one unreadable measure must not abort a sweep
        return {**row, "verdict": "N/A", "note": f"{type(exc).__name__}: {exc}"}

    row["metric"] = metric_a if metric_a == metric_b else f"{metric_a} vs {metric_b}!"
    if metric_a != metric_b:
        return {**row, "verdict": "N/A", "note": "incomparable metrics"}

    common = sorted(set(a.index) & set(b.index))
    if not common:
        return {**row, "verdict": "N/A", "note": "no shared random seeds"}
    a, b = a.loc[common], b.loc[common]

    row.update({"n_seeds": len(common),
                "mean_A": a.mean(), "sd_A": a.std(),
                "mean_B": b.mean(), "sd_B": b.std(),
                "abs_diff": b.mean() - a.mean(),
                "pct_diff": (b.mean() - a.mean()) / a.mean() * 100 if a.mean() else np.nan,
                "seeds_exact": int((a.values == b.values).sum())})
    row["shift_in_sd"] = abs(row["abs_diff"]) / a.std() if a.std() else np.nan

    if np.array_equal(a.values, b.values):
        return {**row, "t": 0.0, "p": 1.0, "verdict": "IDENTICAL"}

    if _scipy_stats is not None and len(common) > 1:
        t, p = _scipy_stats.ttest_ind(b.values, a.values, equal_var=False)
    else:
        n, se = len(common), np.sqrt(a.var(ddof=1) / len(common) + b.var(ddof=1) / len(common))
        t, p = ((b.mean() - a.mean()) / se if se else np.nan), np.nan
    row["t"], row["p"] = t, p
    at = abs(t)
    row["verdict"] = "DIFFERS" if at >= T_DIFFERS else "REVIEW" if at >= T_REVIEW else "OK"
    return row


VERDICT_ORDER = {"DIFFERS": 0, "REVIEW": 1, "OK": 2, "IDENTICAL": 3, "N/A": 4}


def compare_all(measures=None, scenario="__default__", run_a=None, run_b=None) -> pd.DataFrame:
    run_a, run_b = run_a or RUN_A, run_b or RUN_B
    if measures is None:
        mb = list_measures(run_b)
        measures = sorted(m for m in list_measures(run_a) if MEASURE_ALIASES.get(m, m) in mb)
    rows = [compare_measure(m, scenario=scenario, run_a=run_a, run_b=run_b) for m in measures]
    df = pd.DataFrame(rows)
    return df.sort_values("verdict", key=lambda s: s.map(VERDICT_ORDER)).reset_index(drop=True)


def style_summary(df: pd.DataFrame):
    """Colour the verdict column. The text label is always present -- never colour alone."""
    cols = [c for c in ("sim", "combo", "measure", "scenario", "metric", "n_seeds",
                        "mean_A", "mean_B", "pct_diff", "shift_in_sd", "t",
                        "seeds_exact", "verdict") if c in df.columns]
    out = df[cols].rename(columns={"mean_A": f"mean [{LABEL_A}]",
                                   "mean_B": f"mean [{LABEL_B}]"})
    return (out.style
            .format({c: "{:,.2f}" for c in out.columns if out[c].dtype.kind == "f"})
            .apply(lambda s: [f"color: {STATUS_COLORS.get(v, INK)}; font-weight: 600"
                              for v in s] if s.name == "verdict" else ["" for _ in s]))

## 4. Sweep — every sim, vehicle and location

One pass over `SIMS` x `COMBOS`. This is the "did anything move anywhere" view; the
sections after it drill into a single pair.

In [ ]:
def sweep(measures=None, sims=None, combos=None, scenario="__default__") -> pd.DataFrame:
    """Compare every (sim, vehicle, location). Missing runs are reported, not fatal."""
    rows = []
    for sim in (sims or SIMS):
        for combo in (combos or COMBOS):
            a, b = run_paths(sim, combo)
            tag = {"sim": sim, "combo": f"{combo[0]}/{combo[1]}"}
            try:
                resolve_results_dir(a), resolve_results_dir(b)
            except FileNotFoundError as exc:
                rows.append({**tag, "measure": "(all)", "verdict": "N/A", "note": str(exc)})
                continue
            for r in compare_all(measures, scenario, run_a=a, run_b=b).to_dict("records"):
                rows.append({**tag, **r})
    df = pd.DataFrame(rows)
    return df.sort_values(["verdict", "sim", "combo"],
                          key=lambda s: s.map(VERDICT_ORDER) if s.name == "verdict" else s
                          ).reset_index(drop=True)


sweep_df = sweep()
counts = sweep_df.groupby(["sim", "combo"]).verdict.value_counts().unstack(fill_value=0)
print("verdict counts by run:")
display(counts)
print(f"\nseeds reproducing exactly: {int(sweep_df.seeds_exact.fillna(0).sum())} "
      f"of {int(sweep_df.n_seeds.fillna(0).sum())} measure-seed pairs")

In [ ]:
# Everything that is not clean, worst first.
flagged = sweep_df[sweep_df.verdict.isin(["DIFFERS", "REVIEW", "N/A"])]
if flagged.empty:
    print("nothing flagged -- every measure is IDENTICAL or within noise")
else:
    display(style_summary(flagged))

## 5. Single-pair detail

In [ ]:
RUN_A, RUN_B = run_paths(DEFAULT_SIM, DEFAULT_COMBO)
print(f"{DEFAULT_SIM} {DEFAULT_COMBO[0]}/{DEFAULT_COMBO[1]}   |   scenario: {SCENARIO or '(all)'}")
summary = compare_all()
print(f"{len(summary)} measures -- "
      f"{(summary.verdict == 'IDENTICAL').sum()} identical, "
      f"{(summary.verdict == 'OK').sum()} within noise, "
      f"{(summary.verdict == 'REVIEW').sum()} to review, "
      f"{(summary.verdict == 'DIFFERS').sum()} differing")
style_summary(summary)

### How big is each shift, relative to that measure's own noise?

`shift_in_sd` = |mean difference| / per-seed SD of run A. Below ~1 the shift is comparable
to the scatter between two seeds of the *same* run; well above it, the distribution moved.

In [ ]:
def plot_shift(summary: pd.DataFrame, ax=None):
    d = summary.dropna(subset=["shift_in_sd"]).sort_values("shift_in_sd")
    if d.empty:
        print("nothing to plot")
        return
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 0.36 * len(d) + 1.4))
    ax.barh(np.arange(len(d)), d.shift_in_sd,
            color=[STATUS_COLORS.get(v, INK_MUTED) for v in d.verdict], height=0.62)
    ax.set_yticks(np.arange(len(d)), d.measure)
    ax.axvline(1.0, color=INK_MUTED, lw=1, ls="--", zorder=0)
    ax.text(1.0, len(d) - 0.35, " 1 SD", color=INK_MUTED, fontsize=9, va="top")
    ax.set_xlabel("|mean difference| / per-seed SD")
    ax.set_title(f"Shift relative to seed noise — {LABEL_B} vs {LABEL_A}", loc="left")
    ax.grid(axis="y", visible=False)
    for yi, (v, pct) in enumerate(zip(d.shift_in_sd, d.pct_diff)):
        ax.text(v, yi, f"  {pct:+.2f}%", va="center", fontsize=9, color=INK_MUTED)
    ax.set_xlim(0, max(d.shift_in_sd.max() * 1.35, 1.25))
    handles = [plt.Line2D([], [], marker="s", ls="", markersize=8,
                          color=STATUS_COLORS[k], label=k)
               for k in ("IDENTICAL", "OK", "REVIEW", "DIFFERS") if k in set(d.verdict)]
    ax.legend(handles=handles, loc="lower right", fontsize=9)
    plt.tight_layout()
    return ax


plot_shift(summary)
plt.show()

### Per-seed detail

If the two clouds interleave, the runs agree distributionally. If every seed sits on the
same side, the shift is systematic — regardless of what the t-statistic says.

In [ ]:
def plot_seeds(measure_a, measure_b=None, scenario="__default__",
               run_a=None, run_b=None, ax=None):
    run_a, run_b = run_a or RUN_A, run_b or RUN_B
    measure_b = measure_b or MEASURE_ALIASES.get(measure_a, measure_a)
    scenario = SCENARIO if scenario == "__default__" else scenario
    a, _ = _per_seed(load_measure(run_a, measure_a), scenario)
    b, _ = _per_seed(load_measure(run_b, measure_b), scenario)
    common = sorted(set(a.index) & set(b.index))
    a, b = a.loc[common], b.loc[common]

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 3.6))
    x = np.arange(len(common))
    for xi, (av, bv) in enumerate(zip(a.values, b.values)):
        ax.plot([xi, xi], [av, bv], color="#d8d6d0", lw=1.5, zorder=0)
    ax.plot(x, a.values, "o", color=COLOR_A, markersize=8, label=LABEL_A)
    ax.plot(x, b.values, "o", color=COLOR_B, markersize=8, label=LABEL_B)
    ax.axhline(a.mean(), color=COLOR_A, lw=1.5, ls="--", alpha=0.7)
    ax.axhline(b.mean(), color=COLOR_B, lw=1.5, ls="--", alpha=0.7)
    ax.set_xticks(x, [str(s) for s in common])
    ax.set_xlabel("random seed")
    ax.set_ylabel(measure_a)
    ax.set_title(f"{measure_a} per seed — scenario: {scenario or '(all)'}", loc="left")
    ax.legend(loc="best", fontsize=9)
    ax.grid(axis="x", visible=False)
    plt.tight_layout()
    return ax


worst = summary.iloc[0]["measure"] if len(summary) else None
if worst:
    plot_seeds(worst)
    plt.show()

## 6. Where inside a measure does a difference sit?

Aggregate agreement can hide offsetting shifts. This breaks one measure down by any
stratification column.

In [ ]:
def compare_strata(measure_a, by, measure_b=None, scenario="__default__",
                   run_a=None, run_b=None, top=25) -> pd.DataFrame:
    run_a, run_b = run_a or RUN_A, run_b or RUN_B
    measure_b = measure_b or MEASURE_ALIASES.get(measure_a, measure_a)
    scenario = SCENARIO if scenario == "__default__" else scenario
    by = [by] if isinstance(by, str) else list(by)

    def agg(run, measure):
        df = load_measure(run, measure)
        col = _scenario_col(df)
        if scenario is not None and col is not None:
            df = df[df[col] == scenario]
        missing = [c for c in by if c not in df.columns]
        if missing:
            raise KeyError(f"{missing} not in {measure} ({sorted(df.columns)})")
        g = df.groupby(by, observed=True)
        return g["value"].sum() if "value" in df.columns else g.size().astype(float)

    out = pd.DataFrame({LABEL_A: agg(run_a, measure_a),
                        LABEL_B: agg(run_b, measure_b)}).fillna(0.0)
    out["abs_diff"] = out[LABEL_B] - out[LABEL_A]
    out["pct_diff"] = np.where(out[LABEL_A] != 0, out["abs_diff"] / out[LABEL_A] * 100, np.nan)
    return out.reindex(out.pct_diff.abs().sort_values(ascending=False).index).head(top)


if worst:
    scol = _scenario_col(load_measure(RUN_A, worst))
    display(compare_strata(worst, scol, scenario=None) if scol
            else f"{worst} has no scenario column")

In [ ]:
# Any stratification column works:
# compare_strata(worst, "age_group")
# compare_strata(worst, ["age_group", "wealth_quintile"], top=15)

## 7. Add your own

Every helper takes optional `run_a` / `run_b`, so nothing is tied to the defaults:

```python
a, b = run_paths("child", ("bouillon", "nigeria"))
compare_all(run_a=a, run_b=b)
compare_measure("deaths", run_a=a, run_b=b)
plot_seeds("deaths", run_a=a, run_b=b)
compare_strata("deaths", "age_group", run_a=a, run_b=b)
sweep(measures=["deaths", "ylls"])          # one measure across every run
compare_requirements(a, b); compare_configuration(a, b)
```

In [ ]:
# Scratch cell.